# W13-D2 概念实验：安全场景为什么误报率是核心挑战

配套阅读材料：`第13周-Day2-SecurityAnalytics-误报率是核心挑战.md`

md 的论断：**误报不是质量瑕疵，是系统杀手**——先验概率太低，任何小于 100% 的精度都会被基数淹没；单阈值无解，必须多层漏斗。本 notebook 用三个可执行实验定量验证：

**实验 1：PPV 崩塌（数学）** — 固定一个"优秀"检测器（TPR=95%, FPR=0.1%），扫描事件先验 π，看告警的可信度（PPV）如何随场景塌方。
**实验 2：三层漏斗（Monte Carlo）** — 模拟一年 52,560 轮截图评估 + 2 起真实火灾，对比"裸阈值 / 调严阈值 / +连续确认 / +冷却"四档配置的误报数与火灾检出数。
**实验 3：信任动力学** — 误报如何把未来的真告警"变相漏报"：模拟 3 年运营，第 30 个月真火警来临时，不同误报水平下告警被当真的概率。

对应代码事实（md §5）：`fire_smoke.py` 的 conf/label/area/ROI 四闸门、`engine.py` 的 `min_stay_seconds`/`cooldown_seconds`、`entities.py` 的 `AlertStatus.false_positive`。

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print("字体就绪:", font_name)

rng = np.random.default_rng(42)

## 实验 1：base rate 淹没精度

同一个检测器（指标完全相同），放到三个场景里：
- ERP 月结异常：先验 5% —— 告警 98% 可信
- 客流越线：先验 0.5% —— 还行
- 商场火灾：10 年 1 次、每 10 分钟评估一轮 —— 单轮先验 ≈ 2×10⁻⁸

**看 PPV 掉到多少。这就是"换更强的模型救不了安全场景"的数学形态。**

In [ ]:
# ═══ 实验 1：base rate 如何淹没精度 —— PPV 崩塌曲线 ═══
# 检测器指标固定为"优秀"：召回 TPR = 0.95，单次评估误报率 FPR = 0.001（千分之一）
# 唯一变量：真实事件的先验概率 π（对数轴扫描）
# 医学筛查同款公式：PPV = TPR·π / (TPR·π + FPR·(1-π))

TPR, FPR = 0.95, 0.001
pi = np.logspace(-7, -0.3, 400)          # 先验从千万分之一扫到 ~50%
ppv = (TPR * pi) / (TPR * pi + FPR * (1 - pi))

# 三个业务场景的先验（每次评估视角）
scenarios = {
    "ERP月结异常 (π=5%)":        0.05,
    "客流越线事件 (π=0.5%)":     0.005,
    "商场火灾 (π≈1/10年×每10分钟评估)": 1 / (10 * 365 * 24 * 6),
}
# 火灾先验换算：10 年 1 次真火警，每 10 分钟评估一轮 → 单轮先验 ≈ 1.9e-8

fig, ax = plt.subplots(figsize=(9, 5.2))
ax.semilogx(pi, ppv, lw=2.2, color="#d62728", label=f"TPR={TPR}, FPR={FPR}")
ax.axhline(0.9, ls="--", c="gray", alpha=0.6)
ax.text(2e-7, 0.915, "PPV=90%（告警基本可信线）", fontsize=9, color="gray")

colors = ["#2ca02c", "#1f77b4", "#d62728"]
offs = [(10, -22), (12, 10), (12, 8)]
for (name, p), c, off in zip(scenarios.items(), colors, offs):
    v = (TPR * p) / (TPR * p + FPR * (1 - p))
    ax.scatter([p], [v], s=90, zorder=5, color=c, edgecolor="k")
    ax.annotate(f"{name}\nPPV={v:.1%}", (p, v), textcoords="offset points",
                xytext=off, fontsize=9, color=c, arrowprops=dict(arrowstyle="->", color=c))
ax.set_xlabel("真实事件先验概率 π（对数轴）")
ax.set_ylabel("告警的正面预测值 PPV（报了之后是真的概率）")
ax.set_title("实验1：同一个『优秀』检测器，先验越低，告警越不可信")
ax.set_ylim(0, 1.15); ax.grid(alpha=0.3); ax.legend(loc="lower right")
plt.tight_layout(); plt.savefig("w13d2_ppv_collapse.png", dpi=110); plt.show()

for name, p in scenarios.items():
    v = (TPR * p) / (TPR * p + FPR * (1 - p))
    print(f"{name:<38s} π={p:.2e}  →  PPV = {v:.3%}")

## 实验 2：单阈值 vs 三层漏斗 —— 两类事件、三个指标

建模要点（对应 MallSenseAI 代码）：
- **噪声是肥尾的但瞬态的**：无火轮置信度独立采样（蒸汽/夕照/反光偶发高分），轮与轮不相关
- **不是所有火都变大**：明火（2起/年）前 4 轮弱信号、后 8 轮强信号；阴燃（6起/年）全程 5 轮弱信号——早期小火/雾状烟雾，不会自己变大。**早期检测的价值就在阴燃阶段抓住它**
- 时间确认（`min_stay_seconds`）的威力与代价同源：**噪声要连中 N 次独立事件（概率指数衰减），但短事件也凑不齐 N 连**
- `cooldown` 治第三类问题：重复告警刷屏

**五档配置 × 三个指标（年误报 / 明火检出+阴燃检出 / 报警延迟）。先猜哪档能同时赢三个角——再跑。**

In [ ]:
# ═══ 实验 2：三层漏斗 Monte Carlo —— 单阈值三难 vs 参数解耦 ═══
# 模拟一个摄像头，每 10 分钟截图评估一轮，一年 ~52,560 轮。
# 两类真实事件（这是关键建模：**不是所有火都变大**）：
#   明火（2起/年）：前 4 轮弱信号 Beta(9,5)，后 8 轮强信号 Beta(40,4)
#   阴燃（6起/年）：全程 5 轮弱信号 Beta(5,6)——早期小火/雾状烟雾，不会自己变大
# 噪声：肥尾 Beta(1,8)——蒸汽/夕照/反光，轮间独立（瞬态）

N_ROUNDS = int(365 * 24 * 6)
FLAMES, SMOLDERS = 2, 6
rng_ = np.random.default_rng(7)
starts_flame = sorted(rng_.choice(np.arange(0, N_ROUNDS - 40), FLAMES, replace=False).astype(int))
starts_smolder = sorted(rng_.choice(np.arange(0, N_ROUNDS - 40), SMOLDERS, replace=False).astype(int))
conf = rng_.beta(1, 8, N_ROUNDS)
events = []                                    # (start, end, kind)
for s in starts_flame:
    conf[s:s+4] = rng_.beta(9, 5, 4); conf[s+4:s+12] = rng_.beta(40, 4, 8)
    events.append((s, s+12, "flame"))
for s in starts_smolder:
    conf[s:s+5] = rng_.beta(5, 6, 5)
    events.append((s, s+5, "smolder"))

def run(threshold=0.5, min_stay=1, cooldown=0):
    """① conf 超阈值 ② 连续 min_stay 轮 ③ 告警后 cooldown 轮静默。"""
    above = conf >= threshold
    alerts, fp, last = [], 0, -10**9
    for i in range(N_ROUNDS):
        if not above[i]: continue
        if min_stay > 1 and not all(above[max(0, i-min_stay+1): i+1]): continue
        if i - last < cooldown: continue
        last = i; alerts.append(i); fp += (not any(a <= i < b for a, b, _ in events))
    caught = {k: 0 for k in ("flame", "smolder")}; delays = []
    for a, b, k in events:
        in_win = [x for x in alerts if a <= x < b]
        if in_win:
            caught[k] += 1
            if k == "flame": delays.append(in_win[0] - a)
    return fp, caught, len(alerts), (np.mean(delays) if delays else None)

configs = {
    "裸阈值 θ=0.5":            dict(threshold=0.5),
    "裸阈值 θ=0.7（调严）":      dict(threshold=0.7),
    "θ=0.5+连续2轮":           dict(threshold=0.5, min_stay=2),
    "θ=0.5+连续3轮":           dict(threshold=0.5, min_stay=3),
    "θ=0.5+连3轮+冷却4h":     dict(threshold=0.5, min_stay=3, cooldown=24),
}
print(f"一年 {N_ROUNDS:,} 轮 | 明火 {FLAMES} 起（12轮，前4弱后8强） | 阴燃 {SMOLDERS} 起（5轮全弱信号）\n")
print(f"{'配置':<22s}{'误报/年':>8s}{'明火检出':>9s}{'阴燃检出':>9s}{'总告警':>7s}{'明火报警延迟':>12s}")
results = {}
for name, kw in configs.items():
    fp, caught, tot, d = run(**kw)
    results[name] = (fp, caught, tot, d)
    ds = f"{d:.1f}轮(≈{d*10:.0f}分)" if d is not None else "—"
    print(f"{name:<24s}{fp:>6d}{caught['flame']:>7d}/{FLAMES}{caught['smolder']:>7d}/{SMOLDERS}{tot:>6d}{ds:>14s}")

names = list(results)
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
plt.subplots_adjust(bottom=0.24, wspace=0.3)
cols = ["#d62728", "#ff7f0e", "#2ca02c", "#1f77b4", "#9467bd"]
axes[0].bar(names, [results[n][0] for n in names], color=cols); axes[0].set_yscale("log")
axes[0].set_ylabel("年误报数（log）"); axes[0].set_title("①误报：谁在狼来了")
for i, n in enumerate(names):
    v = results[n][0]
    axes[0].text(i, max(v, 0.7) * 1.2, str(v), ha="center", fontsize=9, fontweight="bold")
x = np.arange(len(names))
axes[1].bar(x - 0.18, [results[n][1]['flame'] for n in names], 0.36, label=f"明火(共{FLAMES})", color="#555")
axes[1].bar(x + 0.18, [results[n][1]['smolder'] for n in names], 0.36, label=f"阴燃(共{SMOLDERS})", color="#c7b04a")
axes[1].set_xticks(x); axes[1].set_xticklabels(names)
axes[1].set_ylabel("检出事件数"); axes[1].set_title("②漏报：调严/加确认都伤阴燃"); axes[1].legend(fontsize=8)
axes[2].bar(names, [results[n][3] if results[n][3] is not None else 0 for n in names], color=cols)
axes[2].set_ylabel("明火首次报警延迟（轮）"); axes[2].set_title("③速度：早期检测价值")
axes[2].axhline(4, ls="--", c="gray", alpha=0.6)
axes[2].text(0.02, 4.15, "强信号阶段开始", fontsize=8, color="gray")
for ax in axes: ax.tick_params(axis="x", rotation=25, labelsize=8)
plt.savefig("w13d2_funnel.png", dpi=110); plt.show()

print("\n观察：没有一档配置同时赢①②③——调严压误报但漏阴燃；加确认同样伤阴燃；")
print("漏斗的价值不是全赢，是把三个目标解耦成三个独立旋钮（θ/min_stay/cooldown），")
print("每个场景（对应 MallSenseAI 的每条规则 config）可以按自己的代价结构单独调。")

## 实验 3：误报的真正代价——杀死未来的真告警

误报率指标的险恶之处：它伤害的不是当下这一次告警，而是**未来真告警的存活率**。
"狼来了"不是比喻，是可以量化的动力学：每次误报折损信任本金，真火警来临时告警被当真的概率 = 当前信任余额。

**对比两条信任曲线在第 30 个月的落差。**

In [ ]:
# ═══ 实验 3：信任动力学 —— 误报如何变相制造漏报 ═══
# 信任模型：每次误报 trust ×= 0.85（狼来了折损），每次被当真的真告警 trust ×= 1.05（封顶1.0）
# 模拟 3 年，第 30 个月发生一起真实火警，观察"告警被当真"的概率。
# 对比两种年误报水平：12 次/年（月 1 次）vs 0.5 次/年（半年 1 次）

MONTHS = 36
FIRE_MONTH = 30
def simulate_trust(fp_per_month):
    trust, traj = 0.95, []
    for m in range(MONTHS):
        fps = rng.poisson(fp_per_month)
        trust *= 0.85 ** fps
        if m == FIRE_MONTH:            # 真告警到来，被当真概率 = 当前信任
            traj.append((m, trust)); break
        trust = min(1.0, trust * 1.005)  # 无事发生，信任缓慢回升
        traj.append((m, trust))
    return traj, trust

fig, ax = plt.subplots(figsize=(9, 5))
for rate, c, lbl in [(1.0, "#d62728", "误报 12次/年（每月1次）"),
                     (0.5/12, "#2ca02c", "误报 0.5次/年（半年1次）")]:
    traj, t_at_fire = simulate_trust(rate)
    xs, ys = zip(*traj)
    ax.plot(xs, ys, color=c, lw=2, label=lbl)
    ax.scatter([FIRE_MONTH], [t_at_fire], color=c, s=120, zorder=5, edgecolor="k")
    ax.annotate(f"真火警来临时\n被当真概率 {t_at_fire:.0%}",
                (FIRE_MONTH, t_at_fire), textcoords="offset points", xytext=(-105, -38),
                fontsize=10, color=c, arrowprops=dict(arrowstyle="->", color=c))
ax.axvline(FIRE_MONTH, ls="--", c="gray", alpha=0.5)
ax.text(FIRE_MONTH + 0.3, 0.5, "真实火警", rotation=90, fontsize=10, color="gray")
ax.set_xlabel("月份"); ax.set_ylabel("系统信任度（告警被当真的概率）")
ax.set_title("实验3：误报不是消耗耐心，是消耗真告警的存活率")
ax.set_ylim(0, 1.12); ax.grid(alpha=0.3); ax.legend(loc="lower left")
plt.tight_layout(); plt.savefig("w13d2_trust.png", dpi=110); plt.show()

## 总结

| 实验 | 验证的论断 | 关键数字 |
|---|---|---|
| 1 PPV 崩塌 | 模型精度 ≠ 有效告警质量 | TPR 95% + FPR 0.1% 的优秀检测器，在火灾先验（≈2×10⁻⁶）下 PPV 掉到 0.2% |
| 2 三难与解耦 | 单阈值把三个目标压进一个数字，永远调不平 | 裸宽=误报多；调严=漏阴燃；+连续确认=也伤阴燃。漏斗的价值 = 参数解耦（θ/min_stay/cooldown 各管一个目标）+ 场景级配置 |
| 3 信任动力学 | 误报变相制造漏报 | 月 1 次误报 vs 半年 1 次误报，真火警来临时被当真概率的差距 |

**与 md §9 的呼应**：误报治理是系统工程（证据维度堆叠 × 时间维度确认 × 社会维度裁决），不是模型选型。
**当前最大 Gap（md §5 末尾）**：漏斗是静态的——`false_positive` 人审标记不回流。若把实验 2 的阈值/时长参数变成随人审反馈自动调整的量，静态漏斗就升级成闭环。

**思考（不运行）**：如果把实验 3 的信任折损系数从 0.85 改成 0.95（更宽容的用户），结论变吗？临界点在哪？